# AL Quant Quality Benchmark

Compares Oxidize **AL5 / AL6 / AL8 / AL5_XS** reconstruction RMSE against **Q4_0** on synthetic blocks.

Run in **Google Colab** (CPU runtime is fine) or locally after `cargo build -p oxidize-quantize --release`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().system("git clone -q --depth 1 https://github.com/oxidize-ai/oxidize.git /content/oxidize 2>/dev/null || true")
    os.chdir("/content/oxidize")
    get_ipython().system("cargo build -p oxidize-quantize --release -q")
    ROOT = Path("/content/oxidize")
else:
    ROOT = Path("..").resolve()

QZ = ROOT / "target/release/oxidize-quantize"
assert QZ.exists(), f"build oxidize-quantize first: {QZ}"
print("quantize binary:", QZ)

In [ ]:
# Run the Rust AL vs Q4_0 quality gate (MSE on gaussian weights)
import subprocess
r = subprocess.run(
    ["cargo", "test", "-p", "oxidize-core", "al5_beats_q4_0", "--release", "--", "--nocapture"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit(r.returncode)
print("AL5 beats Q4_0 on block MSE — see oxidize-core/src/compute/quantization/tests.rs for details.")